# Gold, live

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adona-Robot/market-data-examples/blob/main/notebooks/gold_live.ipynb)

Opens the live stream of the [adona-robot](https://adona-robot.com/en) API for gold against the dollar: each bid and ask as it arrives, and one-second bars as they close, for thirty seconds.

You need an API key: the [free trial](https://adona-robot.com/en/signup) carries XAUUSD. On subscribe, the candle channel first sends its recent bars, billed like a REST page.

In [ ]:
%pip install -q "adona-market-data[all]"

In [ ]:
import os
from getpass import getpass

# The API key, from Colab's secrets (key icon on the left, name ADONA_API_KEY) or typed here.
# It stays in this session: never paste it into a cell's text.
try:
    from google.colab import userdata
    os.environ["ADONA_API_KEY"] = userdata.get("ADONA_API_KEY")
except Exception:
    if not os.environ.get("ADONA_API_KEY"):
        os.environ["ADONA_API_KEY"] = getpass("API key (adk_...): ")

In [ ]:
import asyncio
from adona_market_data import Bar, Client, Quote

client = Client(end_user_id="colab")
quotes, bars = [], []


async def listen(seconds: float):
    async def collect():
        async for event in client.stream(["quote:XAUUSD", "candle:XAUUSD:S1"]):
            if isinstance(event, Quote):
                quotes.append(event)
                print(event.as_of.time(), "bid", event.bid, "ask", event.ask)
            elif isinstance(event, Bar) and not event.snapshot:
                bars.append(event.candle)

    try:
        await asyncio.wait_for(collect(), seconds)
    except asyncio.TimeoutError:
        pass  # leaving the loop closes the socket


await listen(30)
print(len(quotes), "quotes and", len(bars), "closed one-second bars")

Outside market hours (weekends, the daily pause) no price moves, so nothing arrives: run it again on a weekday.

## As a DataFrame

In [ ]:
import pandas as pd

spread = pd.Series([float(q.ask - q.bid) for q in quotes], index=[q.as_of for q in quotes], name="ask - bid")
spread.describe()

## Further

- The stream reconnects by itself and stops, raising `StreamStopped`, on what reconnecting cannot change, such as an ended trial.
- [A gold price WebSocket API](https://adona-robot.com/en/use/live-gold-price-api), the guide behind this notebook.